# 08 Daily Schedule Watchlist

Generate a schedule-aware daily MLB hitter watchlist by combining MLB schedule/probable starters, Ridge predictions, batter weakness profiles, and opposing pitcher pitch mix.


In [1]:
import sys
from pathlib import Path

import pandas as pd
import numpy as np
import statsapi
from pybaseball import playerid_reverse_lookup

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(PROJECT_ROOT))

from src.data_loader import save_csv

print(PROJECT_ROOT)


/Users/henry_tsai/Desktop/daily-mlb-hitter-forecasting/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


/Users/henry_tsai/Desktop/daily-mlb-hitter-forecasting


## 1. Configuration


In [2]:
target_date = "2025-05-15"

team_name_to_abbr = {
    "Arizona Diamondbacks": "AZ",
    "Atlanta Braves": "ATL",
    "Baltimore Orioles": "BAL",
    "Boston Red Sox": "BOS",
    "Chicago Cubs": "CHC",
    "Chicago White Sox": "CWS",
    "Cincinnati Reds": "CIN",
    "Cleveland Guardians": "CLE",
    "Colorado Rockies": "COL",
    "Detroit Tigers": "DET",
    "Houston Astros": "HOU",
    "Kansas City Royals": "KC",
    "Los Angeles Angels": "LAA",
    "Los Angeles Dodgers": "LAD",
    "Miami Marlins": "MIA",
    "Milwaukee Brewers": "MIL",
    "Minnesota Twins": "MIN",
    "New York Mets": "NYM",
    "New York Yankees": "NYY",
    "Oakland Athletics": "OAK",
    "Athletics": "OAK",
    "Philadelphia Phillies": "PHI",
    "Pittsburgh Pirates": "PIT",
    "San Diego Padres": "SD",
    "San Francisco Giants": "SF",
    "Seattle Mariners": "SEA",
    "St. Louis Cardinals": "STL",
    "Tampa Bay Rays": "TB",
    "Texas Rangers": "TEX",
    "Toronto Blue Jays": "TOR",
    "Washington Nationals": "WSH",
}


## 2. Retrieve MLB schedule and probable starters


In [3]:
raw_schedule = statsapi.get(
    "schedule",
    {
        "sportId": 1,
        "date": target_date,
        "hydrate": "probablePitcher",
    },
)

if not raw_schedule.get("dates"):
    raise ValueError(f"No MLB games found for {target_date}")

games_raw = raw_schedule["dates"][0]["games"]

schedule_rows = []
for g in games_raw:
    teams = g["teams"]
    away = teams["away"]
    home = teams["home"]

    away_probable = away.get("probablePitcher", {})
    home_probable = home.get("probablePitcher", {})

    schedule_rows.append({
        "game_id": g.get("gamePk"),
        "game_date": g.get("gameDate"),
        "away_team": away["team"]["name"],
        "away_team_id": away["team"]["id"],
        "home_team": home["team"]["name"],
        "home_team_id": home["team"]["id"],
        "away_probable_pitcher": away_probable.get("fullName"),
        "away_probable_pitcher_id": away_probable.get("id"),
        "home_probable_pitcher": home_probable.get("fullName"),
        "home_probable_pitcher_id": home_probable.get("id"),
    })

schedule_df = pd.DataFrame(schedule_rows)
schedule_df


,game_id,game_date,away_team,away_team_id,home_team,home_team_id,away_probable_pitcher,away_probable_pitcher_id,home_probable_pitcher,home_probable_pitcher_id
0,777909,2025-05-15T16:15:00Z,Washington Nationals,120,Atlanta Braves,144,Trevor Williams,592866,AJ Smith-Shawver,700363
1,777911,2025-05-15T16:35:00Z,Minnesota Twins,142,Baltimore Orioles,110,Chris Paddack,663978,Tomoyuki Sugano,608372
2,777912,2025-05-15T16:40:00Z,Chicago White Sox,145,Cincinnati Reds,113,Bryse Wilson,669060,Nick Martinez,607259
3,777913,2025-05-15T19:07:00Z,Tampa Bay Rays,139,Toronto Blue Jays,141,Zack Littell,641793,Kevin Gausman,592332
4,777910,2025-05-16T00:05:00Z,Houston Astros,117,Texas Rangers,140,Hunter Brown,686613,Jacob deGrom,594798
5,777946,2025-05-16T02:10:00Z,Athletics,133,Los Angeles Dodgers,119,Osvaldo Bido,674370,Matt Sauer,669422


In [4]:
team_games = []

for _, row in schedule_df.iterrows():
    # Away hitters face home probable pitcher
    team_games.append({
        "team": row["away_team"],
        "opponent_team": row["home_team"],
        "opposing_probable_pitcher": row["home_probable_pitcher"],
        "opposing_pitcher_id": row["home_probable_pitcher_id"],
        "game_id": row["game_id"],
        "game_date": row["game_date"],
        "home_away": "away",
    })

    # Home hitters face away probable pitcher
    team_games.append({
        "team": row["home_team"],
        "opponent_team": row["away_team"],
        "opposing_probable_pitcher": row["away_probable_pitcher"],
        "opposing_pitcher_id": row["away_probable_pitcher_id"],
        "game_id": row["game_id"],
        "game_date": row["game_date"],
        "home_away": "home",
    })

team_games_df = pd.DataFrame(team_games)

team_games_df["team_abbr"] = team_games_df["team"].map(team_name_to_abbr)
team_games_df["opponent_team_abbr"] = team_games_df["opponent_team"].map(team_name_to_abbr)

missing_team_map = team_games_df[team_games_df["team_abbr"].isna()]["team"].unique()
if len(missing_team_map) > 0:
    print("Warning: missing team abbreviation mapping:", missing_team_map)

team_games_df.head()


,team,opponent_team,opposing_probable_pitcher,opposing_pitcher_id,game_id,game_date,home_away,team_abbr,opponent_team_abbr
0,Washington Nationals,Atlanta Braves,AJ Smith-Shawver,700363,777909,2025-05-15T16:15:00Z,away,WSH,ATL
1,Atlanta Braves,Washington Nationals,Trevor Williams,592866,777909,2025-05-15T16:15:00Z,home,ATL,WSH
2,Minnesota Twins,Baltimore Orioles,Tomoyuki Sugano,608372,777911,2025-05-15T16:35:00Z,away,MIN,BAL
3,Baltimore Orioles,Minnesota Twins,Chris Paddack,663978,777911,2025-05-15T16:35:00Z,home,BAL,MIN
4,Chicago White Sox,Cincinnati Reds,Nick Martinez,607259,777912,2025-05-15T16:40:00Z,away,CWS,CIN


## 3. Load PA data and identify hitter candidates by team


In [5]:
pa_path = PROJECT_ROOT / "data/processed/pa_2025_04_01_to_05_31.csv"

pa_df = pd.read_csv(pa_path)
pa_df["game_date"] = pd.to_datetime(pa_df["game_date"])

required_pa_cols = ["game_date", "batter", "inning_topbot", "home_team", "away_team"]
missing_pa_cols = [c for c in required_pa_cols if c not in pa_df.columns]
if missing_pa_cols:
    raise KeyError(f"Missing required PA columns: {missing_pa_cols}")

pa_df["batting_team"] = np.where(
    pa_df["inning_topbot"] == "Top",
    pa_df["away_team"],
    pa_df["home_team"],
)

pa_df[["game_date", "batter", "batting_team", "home_team", "away_team", "inning_topbot"]].head()


,game_date,batter,batting_team,home_team,away_team,inning_topbot
0,2025-05-31,691785,BOS,ATL,BOS,Top
1,2025-05-31,665966,BOS,ATL,BOS,Top
2,2025-05-31,677800,BOS,ATL,BOS,Top
3,2025-05-31,663897,ATL,ATL,BOS,Bot
4,2025-05-31,671739,ATL,ATL,BOS,Bot


In [6]:
# Use only information available up to the target date
pa_until_target = pa_df[pa_df["game_date"] <= pd.Timestamp(target_date)].copy()

latest_batter_team = (
    pa_until_target
    .sort_values("game_date")
    .groupby("batter")
    .tail(1)
    [["batter", "batting_team"]]
    .rename(columns={"batting_team": "team_abbr"})
)

hitter_candidates = latest_batter_team.merge(
    team_games_df,
    on="team_abbr",
    how="inner",
)

print("latest_batter_team:", latest_batter_team.shape)
print("hitter_candidates:", hitter_candidates.shape)
hitter_candidates.head()


latest_batter_team: (503, 2)
hitter_candidates: (193, 10)


,batter,team_abbr,team,opponent_team,opposing_probable_pitcher,opposing_pitcher_id,game_id,game_date,home_away,opponent_team_abbr
0,644433,ATL,Atlanta Braves,Washington Nationals,Trevor Williams,592866,777909,2025-05-15T16:15:00Z,home,WSH
1,676356,TB,Tampa Bay Rays,Toronto Blue Jays,Kevin Gausman,592332,777913,2025-05-15T19:07:00Z,away,TOR
2,676439,LAD,Los Angeles Dodgers,Athletics,Osvaldo Bido,674370,777946,2025-05-16T02:10:00Z,home,OAK
3,686676,CWS,Chicago White Sox,Cincinnati Reds,Nick Martinez,607259,777912,2025-05-15T16:40:00Z,away,CIN
4,643565,CWS,Chicago White Sox,Cincinnati Reds,Nick Martinez,607259,777912,2025-05-15T16:40:00Z,away,CIN


## 4. Merge model predictions


In [7]:
pred_path = PROJECT_ROOT / "data/predictions/model_predictions_2025_04_01_to_05_31.csv"

pred_df = pd.read_csv(pred_path)
pred_df["prediction_date"] = pd.to_datetime(pred_df["prediction_date"])

pred_for_date = pred_df[pred_df["prediction_date"] == pd.Timestamp(target_date)].copy()

print("pred_for_date:", pred_for_date.shape)
pred_for_date.head()


pred_for_date: (329, 9)


,batter,prediction_date,future_7d_xwOBA,pred_season_xwOBA,pred_last_7d_xwOBA,pred_last_14d_xwOBA,pred_ridge,pred_random_forest,pred_lightgbm
1335,457705,2025-05-15,0.313030,0.355497,0.317064,0.334966,0.332068,0.332762,0.348571
1336,457759,2025-05-15,0.331722,0.310547,0.389374,0.371283,0.313400,0.298965,0.293523
1337,467793,2025-05-15,0.376473,0.293519,0.289103,0.346776,0.316724,0.298830,0.297483
1338,500743,2025-05-15,0.126600,0.291621,0.348136,0.249039,0.294140,0.288429,0.299039
1339,502054,2025-05-15,0.194600,0.244750,0.317999,0.289499,0.291583,0.304692,0.311717


In [8]:
daily_watchlist = hitter_candidates.merge(
    pred_for_date,
    on="batter",
    how="inner",
)

print("daily_watchlist after prediction merge:", daily_watchlist.shape)
daily_watchlist.head()


daily_watchlist after prediction merge: (121, 18)


,batter,team_abbr,team,opponent_team,opposing_probable_pitcher,opposing_pitcher_id,game_id,game_date,home_away,opponent_team_abbr,prediction_date,future_7d_xwOBA,pred_season_xwOBA,pred_last_7d_xwOBA,pred_last_14d_xwOBA,pred_ridge,pred_random_forest,pred_lightgbm
0,678554,TB,Tampa Bay Rays,Toronto Blue Jays,Kevin Gausman,592332,777913,2025-05-15T19:07:00Z,away,TOR,2025-05-15,0.514044,0.286421,0.218378,0.247526,0.295295,0.301076,0.334122
1,650490,TB,Tampa Bay Rays,Toronto Blue Jays,Kevin Gausman,592332,777913,2025-05-15T19:07:00Z,away,TOR,2025-05-15,0.414635,0.334983,0.308028,0.366603,0.339459,0.318986,0.317917
2,671277,WSH,Washington Nationals,Atlanta Braves,AJ Smith-Shawver,700363,777909,2025-05-15T16:15:00Z,away,ATL,2025-05-15,0.304919,0.343356,0.385528,0.339983,0.330962,0.324485,0.278754
3,670770,CIN,Cincinnati Reds,Chicago White Sox,Bryse Wilson,669060,777912,2025-05-15T16:40:00Z,home,CWS,2025-05-15,0.275519,0.296487,0.388061,0.350808,0.309916,0.300609,0.295515
4,663898,HOU,Houston Astros,Texas Rangers,Jacob deGrom,594798,777910,2025-05-16T00:05:00Z,away,TEX,2025-05-15,0.177681,0.302719,0.141687,0.301883,0.311446,0.308736,0.304882


## 5. Add player names


In [9]:
batter_ids = (
    daily_watchlist["batter"]
    .dropna()
    .astype(int)
    .unique()
)

batter_lookup = playerid_reverse_lookup(batter_ids, key_type="mlbam")

batter_name_map = batter_lookup.copy()
batter_name_map["batter_name"] = (
    batter_name_map["name_first"].fillna("") + " " + batter_name_map["name_last"].fillna("")
).str.strip()

batter_name_map = (
    batter_name_map[["key_mlbam", "batter_name"]]
    .rename(columns={"key_mlbam": "batter"})
)

batter_name_map["batter"] = batter_name_map["batter"].astype(int)

daily_watchlist = daily_watchlist.merge(
    batter_name_map,
    on="batter",
    how="left",
)

daily_watchlist[["batter", "batter_name"]].head()


Gathering player lookup table. This may take a moment.


,batter,batter_name
0,678554,curtis mead
1,650490,yandy díaz
2,671277,luis garcía
3,670770,tj friedl
4,663898,brendan rodgers


## 6. Merge batter weakness and opposing pitcher pitch mix


In [10]:
weakness_path = PROJECT_ROOT / "data/features/batter_weakness_profiles_2025_04_01_to_05_31.csv"
pitcher_mix_path = PROJECT_ROOT / "data/features/pitcher_pitch_mix_2025_04_01_to_05_31.csv"

batter_weakness = pd.read_csv(weakness_path)
pitcher_mix = pd.read_csv(pitcher_mix_path)

# Remove duplicate columns if this cell is rerun
cols_to_drop = [
    c for c in daily_watchlist.columns
    if c.startswith("weakness_vs_") 
    or c.startswith("xwOBA_vs_") 
    or c.startswith("PA_vs_")
    or c.startswith("pitcher_usage_")
    or c in ["pitcher", "main_weakness_pitch_group", "main_weakness_score",
             "main_weakness_xwOBA", "main_weakness_league_xwOBA", "main_weakness_PA"]
]
daily_watchlist = daily_watchlist.drop(columns=cols_to_drop, errors="ignore")

daily_watchlist = daily_watchlist.merge(
    batter_weakness,
    on="batter",
    how="left",
)

daily_watchlist = daily_watchlist.merge(
    pitcher_mix,
    left_on="opposing_pitcher_id",
    right_on="pitcher",
    how="left",
)

usage_cols = [c for c in daily_watchlist.columns if c.startswith("pitcher_usage_")]
print("usage columns:", usage_cols)
daily_watchlist[["batter", "batter_name", "team", "opposing_probable_pitcher", "opposing_pitcher_id"] + usage_cols[:5]].head()


usage columns: ['pitcher_usage_changeup', 'pitcher_usage_curveball', 'pitcher_usage_cutter', 'pitcher_usage_fastball', 'pitcher_usage_other', 'pitcher_usage_sinker', 'pitcher_usage_slider', 'pitcher_usage_unknown']


,batter,batter_name,team,opposing_probable_pitcher,opposing_pitcher_id,pitcher_usage_changeup,pitcher_usage_curveball,pitcher_usage_cutter,pitcher_usage_fastball,pitcher_usage_other
0,678554,curtis mead,Tampa Bay Rays,Kevin Gausman,592332,0.344828,0.000000,0.000000,0.581897,0.0
1,650490,yandy díaz,Tampa Bay Rays,Kevin Gausman,592332,0.344828,0.000000,0.000000,0.581897,0.0
2,671277,luis garcía,Washington Nationals,AJ Smith-Shawver,700363,0.405882,0.070588,0.000000,0.488235,0.0
3,670770,tj friedl,Cincinnati Reds,Bryse Wilson,669060,0.196532,0.179191,0.277457,0.109827,0.0
4,663898,brendan rodgers,Houston Astros,Jacob deGrom,594798,0.092511,0.008811,0.000000,0.475771,0.0


## 7. Compute matchup score and signal label


In [11]:
pitch_groups = ["fastball", "slider", "curveball", "changeup", "cutter", "sinker"]

def compute_matchup_weakness_score(row, pitch_groups):
    score = 0.0
    valid = False

    for pitch in pitch_groups:
        weakness_col = f"weakness_vs_{pitch}"
        usage_col = f"pitcher_usage_{pitch}"

        weakness = row.get(weakness_col, np.nan)
        usage = row.get(usage_col, np.nan)

        if pd.notna(weakness) and pd.notna(usage):
            score += weakness * usage
            valid = True

    return score if valid else np.nan


def label_daily_matchup(score):
    if pd.isna(score):
        return "Unknown"
    if score >= 0.04:
        return "Unfavorable"
    if score <= -0.04:
        return "Favorable"
    return "Neutral"


def assign_signal(row):
    pred = row["pred_ridge"]
    season = row["pred_season_xwOBA"]
    last_14d = row["pred_last_14d_xwOBA"]
    matchup = row["matchup_label"]

    if pred >= 0.360 and matchup in ["Favorable", "Neutral"]:
        return "Hot Candidate"

    if pred > season + 0.03 and pred > last_14d + 0.03 and matchup == "Favorable":
        return "Bounce-back Candidate"

    if pred < last_14d - 0.04 or matchup == "Unfavorable":
        return "Regression Risk"

    return "Neutral"


daily_watchlist["matchup_weakness_score"] = daily_watchlist.apply(
    lambda row: compute_matchup_weakness_score(row, pitch_groups),
    axis=1,
)

daily_watchlist["matchup_label"] = daily_watchlist["matchup_weakness_score"].apply(label_daily_matchup)
daily_watchlist["signal"] = daily_watchlist.apply(assign_signal, axis=1)

daily_watchlist["matchup_label"].value_counts(dropna=False), daily_watchlist["signal"].value_counts(dropna=False)


(matchup_label
 Neutral        73
 Unfavorable    25
 Favorable      23
 Name: count, dtype: int64,
 signal
 Neutral                  71
 Regression Risk          47
 Bounce-back Candidate     2
 Hot Candidate             1
 Name: count, dtype: int64)

## 8. Display and save final watchlist


In [12]:
display_cols = [
    "game_date",
    "team",
    "team_abbr",
    "opponent_team",
    "opponent_team_abbr",
    "home_away",
    "batter_name",
    "batter",
    "opposing_probable_pitcher",
    "opposing_pitcher_id",
    "pred_ridge",
    "pred_season_xwOBA",
    "pred_last_14d_xwOBA",
    "matchup_weakness_score",
    "matchup_label",
    "signal",
]

missing_display_cols = [c for c in display_cols if c not in daily_watchlist.columns]
if missing_display_cols:
    raise KeyError(f"Missing display columns: {missing_display_cols}")

daily_watchlist_output = (
    daily_watchlist[display_cols]
    .sort_values(["signal", "pred_ridge"], ascending=[True, False])
)

daily_watchlist_output.head(50)


,game_date,team,team_abbr,opponent_team,opponent_team_abbr,home_away,batter_name,batter,opposing_probable_pitcher,opposing_pitcher_id,pred_ridge,pred_season_xwOBA,pred_last_14d_xwOBA,matchup_weakness_score,matchup_label,signal
26,2025-05-15T16:40:00Z,Chicago White Sox,CWS,Cincinnati Reds,CIN,away,tim elko,671284,Nick Martinez,607259,0.294006,0.211049,0.211049,-0.059725,Favorable,Bounce-back Candidate
9,2025-05-15T19:07:00Z,Toronto Blue Jays,TOR,Tampa Bay Rays,TB,home,jonatan clase,682729,Zack Littell,641793,0.274444,0.164526,0.188600,-0.051404,Favorable,Bounce-back Candidate
56,2025-05-16T02:10:00Z,Los Angeles Dodgers,LAD,Athletics,OAK,home,shohei ohtani,660271,Osvaldo Bido,674370,0.373166,0.472793,0.576910,-0.153015,Favorable,Hot Candidate
63,2025-05-15T16:15:00Z,Atlanta Braves,ATL,Washington Nationals,WSH,home,marcell ozuna,542303,Trevor Williams,592866,0.354785,0.421419,0.378018,-0.083336,Favorable,Neutral
74,2025-05-15T16:15:00Z,Atlanta Braves,ATL,Washington Nationals,WSH,home,matt olson,621566,Trevor Williams,592866,0.354717,0.353972,0.284963,-0.073103,Favorable,Neutral
117,2025-05-15T16:35:00Z,Baltimore Orioles,BAL,Minnesota Twins,MIN,home,gunnar henderson,683002,Chris Paddack,663978,0.350673,0.326076,0.368428,0.004365,Neutral,Neutral
44,2025-05-15T19:07:00Z,Toronto Blue Jays,TOR,Tampa Bay Rays,TB,home,addison barger,680718,Zack Littell,641793,0.348873,0.328562,0.329016,-0.071229,Favorable,Neutral
120,2025-05-15T16:35:00Z,Minnesota Twins,MIN,Baltimore Orioles,BAL,away,trevor larnach,663616,Tomoyuki Sugano,608372,0.348254,0.325925,0.336923,-0.008121,Neutral,Neutral
73,2025-05-15T16:15:00Z,Atlanta Braves,ATL,Washington Nationals,WSH,home,austin riley,663586,Trevor Williams,592866,0.346896,0.350799,0.361070,-0.029586,Neutral,Neutral
36,2025-05-15T19:07:00Z,Toronto Blue Jays,TOR,Tampa Bay Rays,TB,home,vladimir guerrero,665489,Zack Littell,641793,0.346668,0.386074,0.363216,-0.089366,Favorable,Neutral


In [13]:
output_path = PROJECT_ROOT / f"data/predictions/daily_watchlist_{target_date}.csv"
save_csv(daily_watchlist_output, output_path)

summary = (
    daily_watchlist_output
    .groupby(["signal", "matchup_label"])
    .agg(
        n_hitters=("batter", "count"),
        avg_pred_ridge=("pred_ridge", "mean"),
        avg_matchup_weakness_score=("matchup_weakness_score", "mean"),
    )
    .reset_index()
    .sort_values(["signal", "matchup_label"])
)

summary_path = PROJECT_ROOT / f"reports/daily_watchlist_summary_{target_date}.csv"
save_csv(summary, summary_path)

lookup_path = PROJECT_ROOT / "data/processed/batter_name_lookup.csv"
save_csv(batter_name_map, lookup_path)

print(output_path, output_path.exists())
print(summary_path, summary_path.exists())
print(lookup_path, lookup_path.exists())

summary


/Users/henry_tsai/Desktop/daily-mlb-hitter-forecasting/data/predictions/daily_watchlist_2025-05-15.csv True
/Users/henry_tsai/Desktop/daily-mlb-hitter-forecasting/reports/daily_watchlist_summary_2025-05-15.csv True
/Users/henry_tsai/Desktop/daily-mlb-hitter-forecasting/data/processed/batter_name_lookup.csv True


,signal,matchup_label,n_hitters,avg_pred_ridge,avg_matchup_weakness_score
0,Bounce-back Candidate,Favorable,2,0.284225,-0.055565
1,Hot Candidate,Favorable,1,0.373166,-0.153015
2,Neutral,Favorable,13,0.334417,-0.071036
3,Neutral,Neutral,58,0.323821,0.001367
4,Regression Risk,Favorable,7,0.337040,-0.086611
5,Regression Risk,Neutral,15,0.316208,-0.005781
6,Regression Risk,Unfavorable,25,0.302929,0.075994


## Summary

This notebook integrates the forecasting model with the MLB schedule API.

For a selected target date, the pipeline:

1. Retrieves MLB games and probable starting pitchers.
2. Converts schedule team names into Statcast team abbreviations.
3. Identifies hitter candidates based on each hitter's latest team before the target date.
4. Merges Ridge-based future 7-day xwOBA predictions.
5. Adds batter names using MLBAM ID lookup.
6. Merges batter pitch-type weakness profiles.
7. Merges opposing probable pitcher's pitch mix.
8. Computes matchup weakness scores.
9. Assigns daily hitter watchlist signal labels.
10. Saves a schedule-aware daily hitter watchlist output.

This extends the historical watchlist demo into a schedule-aware daily watchlist prototype.
